## 1. Read Data for Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

In [ ]:
import plotly.express as px

In [ ]:
import os 

In [ ]:
os.listdir(r'/Uber/archive')

In [ ]:
uber_15 = pd.read_csv(r'/Uber/archive/uber-raw-data-janjune-15.csv')

In [ ]:
uber_15

In [ ]:
type(uber_15)

## 2. Perform Data Pre-Processing/Data Cleaning


In [ ]:
uber_15.shape

In [ ]:
uber_15[uber_15.duplicated(keep=False)].sort_values(by = uber_15.columns.tolist())

In [ ]:
uber_15.drop_duplicates(inplace=True)

In [ ]:
uber_15.duplicated().sum

In [ ]:
uber_15.shape

In [ ]:
uber_15.isnull().sum()

In [ ]:
uber_15.dtypes

In [ ]:
uber_15['Pickup_date'] = pd.to_datetime(uber_15['Pickup_date'])

## 3. Which Months have max. Uber Pickups in NYC 


In [ ]:
uber_15['month'] = uber_15['Pickup_date'].dt.month_name()
uber_15['weekday'] = uber_15['Pickup_date'].dt.day_name()
uber_15['day'] = uber_15['Pickup_date'].dt.day
uber_15['hour'] = uber_15['Pickup_date'].dt.hour
uber_15['minute'] = uber_15['Pickup_date'].dt.minute

In [ ]:
uber_15['month'].value_counts().plot(kind = 'bar')

In [ ]:
uber_15.head(4)

In [ ]:
pivot = pd.crosstab(index = uber_15['month'], columns = uber_15['weekday'])

In [ ]:
month_order = ['January', 'February', 'March', 'April', 'May', 'June']

#Define the order of weekdays
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

In [ ]:
pivot_sorted = pivot.reindex(index = month_order).reindex(columns = weekday_order)

In [ ]:
import warnings
from warnings import filterwarnings
filterwarnings('ignore')

In [ ]:
fig = px.bar(pivot_sorted, 
      x = pivot_sorted.index ,
      y = pivot_sorted.columns,
      barmode = 'group',
      title = 'Uber pickups by Month & weekday')

In [ ]:
fig.update_layout(
    xaxis_title = 'Month',
    yaxis_title = 'Number of Pickups',
    legend_title = 'week'
)

## 4. Find out the Hourly Rush in NYC on all days

In [ ]:
uber_15.columns

In [ ]:
hourly_rush = uber_15.groupby(['weekday', 'hour'], as_index = False).size()

In [ ]:
hourly_rush.sort_values(['weekday', 'hour'])

In [ ]:
hourly_rush['weekday'] = pd.Categorical(hourly_rush['weekday'], categories=weekday_order, ordered = True)

In [ ]:
hourly_rush = hourly_rush.sort_values(['weekday', 'hour'])
hourly_rush

In [ ]:
hourly_fig = px.line(hourly_rush,
       x = 'hour',
       y= 'size',
       color = 'weekday',
       markers = True,
       title = 'Hourly Uber Rush by days')

In [ ]:
hourly_fig.update_layout(
    xaxis_title = 'hours of day',
    yaxis_title = 'Number of pickups',
    legend_title = 'weekday',
    template = 'plotly_white'
    
)

## 5. Pareto Analysis

In [ ]:
base_counts = uber_15['Dispatching_base_num'].value_counts()

In [ ]:
(base_counts / base_counts.sum()).cumsum()

In [ ]:
base_df = base_counts.reset_index()

In [ ]:
base_df.columns = ['Base', 'Trips']
base_df

In [ ]:
base_df['Trip_Percentage'] = base_df['Trips']/ base_df['Trips'].sum()
base_df

In [ ]:
base_df['Cumulative_percentage'] = base_df['Trip_Percentage'].cumsum()

In [ ]:
import plotly.graph_objs as go

In [ ]:
pareto = go.Figure([
    go.Bar(x = base_df['Base'], y = base_df['Trips'], name = 'Trips'),
    go.Scatter(x = base_df['Base'], y = base_df['Cumulative_percentage'], 
               yaxis = 'y2',
               mode = 'lines+markers' ,
               name = 'Cumulative %')
])

In [ ]:
pareto = pareto.update_layout(yaxis2 = dict(overlaying = 'y', side = 'right', tickformat = '.0%', range = [0,1]))
pareto.add_hline(y=0.8, yref = 'y2', line_dash = 'dash')
pareto

## 6. Airport Demand Analysis 

In [ ]:
uber_15['locationID'].unique()

In [ ]:
airports_ids = [1, 132, 138]

In [ ]:
airport_df = uber_15[uber_15['locationID'].isin(airports_ids)]

In [ ]:
airport_map = {
    1 : 'EWR',
    132 : 'JFK',
    138 : 'LGA'
}

In [ ]:
airport_df['airport_name'] = airport_df['locationID'].map(airport_map)

In [ ]:
airport_df['airport_name'].value_counts()

In [ ]:
airport_df['hour'] = airport_df['Pickup_date'].dt.hour

In [ ]:
airport_df.columns


In [ ]:
airport_hourly = airport_df.groupby(['airport_name', 'hour']).size().reset_index(name = 'trips')
airport_hourly

In [ ]:
area = px.area(airport_hourly,
        x = 'hour',
        y = 'trips',
        color = 'airport_name',
        title = 'Airport Demand Pressure'
       )

## 7. Collect entire Data & Make it ready for the Data Analysis 

In [ ]:
import os 

In [ ]:
files = os.listdir('/Users/adigonitodoulou/Desktop/Data_Analysis_Projects/Uber/archive')


In [ ]:
files 

In [ ]:
files_csv = ['uber-raw-data-apr14.csv',
 'uber-raw-data-aug14.csv',
 'uber-raw-data-sep14.csv',
 'uber-raw-data-jul14.csv',
'uber-raw-data-jun14.csv',
'uber-raw-data-may14.csv']

In [ ]:
final = pd.DataFrame()
path = '/Users/adigonitodoulou/Desktop/Data_Analysis_Projects/Uber/archive'

for file in files_csv:
    current_df = pd.read_csv(path + '/' + file)
    final = pd.concat([current_df, final])

In [ ]:
final.shape

In [ ]:
final.duplicated().sum()

In [ ]:
final.drop_duplicates(inplace=True)

In [ ]:
final.shape

In [ ]:
final.head(3)

## 8. Data Loading: CSV vs JSON vs Database vs Big Data 

In [ ]:
final.to_csv('/Users/adigonitodoulou/Desktop/Data_Analysis_Projects/Uber/Preprocessed_Data/uber_full_data.csv', index=False)

In [ ]:
final.reset_index().to_json('/Users/adigonitodoulou/Desktop/Data_Analysis_Projects/Uber/Preprocessed_Data/uber.json')

In [ ]:
!pip install sqlalchemy

In [ ]:
from sqlalchemy import create_engine

In [ ]:
engine = create_engine('sqlite:////Users/adigonitodoulou/Desktop/Data_Analysis_Projects/Uber/Preprocessed_Data/uber_db.sqlite')

In [ ]:
final.to_sql('Trips', con = engine, if_exists = 'append')

## 9. Hourly Rush via Animated Spatial Analysis

In [ ]:
final.dtypes

In [ ]:
final['Date/Time'] = pd.to_datetime(final['Date/Time'])

In [ ]:
final['hour'] = final['Date/Time'].dt.hour

In [ ]:
final.columns

In [ ]:
hourly_data = []

for h in range(24):
    temp = final[final['hour'] == h][['Lat', 'Lon']]
    hourly_data.append(temp.values.tolist())

In [ ]:
!pip install folium

In [ ]:
import folium 
from folium.plugins import HeatMapWithTime

In [ ]:
m = folium.Map(
    location = [40.7128, -74.0060] , 
    zoom_start = 11,
    tiles= 'OpenStreetMap' 
)


In [ ]:
HeatMapWithTime(hourly_data,
               radius = 9,
               auto_play = False,
               max_opacity=0.8).add_to(m)

In [ ]:
m
